In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
import pickle
warnings.filterwarnings('ignore')

In [ ]:
with open('/content/appml-assignment1-dataset-v2.pkl','rb') as f:
    data=pickle.load(f)
df_x=data['X'][data['X'].columns[:5]]
data['y'].name="target"
df_y=data['y']
df=pd.concat([df_x,df_y],axis=1)

In [ ]:
df.set_index('date',inplace=True)
df.sort_index(inplace=True)

In [ ]:
df['day_of_week']=df.index.dayofweek
df['hour_of_day']=df.index.hour

In [ ]:
df.head()

In [ ]:
x=df[df.columns[:4].values.tolist()+["day_of_week","hour_of_day"]]
y=df['target']

In [ ]:
train_sampels_size=31712

# x_train=x.iloc[:train_sampels_size]
# x_test=x.iloc[train_sampels_size:]
y_train=y.iloc[:train_sampels_size]
y_test=y.iloc[train_sampels_size:]


In [ ]:
final_x_train_data=[]
final_y_train_data=[]
final_x_test_data=[]
final_y_test_data=[]
win_size=4

for i in range(len(y_train)-win_size):
    final_x_train_data.append(y_train.iloc[i:i+win_size].values.tolist())
    final_y_train_data.append(y_train.iloc[i+win_size])
for i in range(len(y_test)-win_size):
    final_x_test_data.append(y_test.iloc[i:i+win_size].values.tolist())
    final_y_test_data.append(y_test.iloc[i+win_size])
final_x_train_data,final_y_train_data=np.expand_dims(np.array(final_x_train_data),axis=-1),np.expand_dims(np.array(final_y_train_data),axis=-1)
final_x_test_data,final_y_test_data=np.expand_dims(np.array(final_x_test_data),axis=-1),np.expand_dims(np.array(final_y_test_data),axis=-1)


In [ ]:
final_x_train_data.shape,final_y_train_data.shape,final_x_test_data.shape,final_y_test_data.shape

In [ ]:
from keras.layers import Input,LSTM,Dense
from keras.callbacks import EarlyStopping
from keras.models import Model

model_input=Input(shape=(4,1))
lstm_output=LSTM(256,return_sequences=False,return_state=False)(model_input)
dense_output=Dense(1,activation="relu")(lstm_output)

model=Model(inputs=[model_input],outputs=[dense_output])


In [ ]:
model.summary()
from keras.utils import plot_model
plot_model(model,show_shapes=True,show_layer_activations=True)

In [ ]:
model.compile(optimizer="adam",loss="mse")

In [ ]:
early_stop=EarlyStopping(monitor="val_loss",patience=5)
model.fit(x=final_x_train_data,y=final_y_train_data,validation_split=0.2,epochs=10,batch_size=128,callbacks=[early_stop])

In [ ]:
final_x_test_data.shape,final_y_test_data.shape

In [ ]:
y_pred=model.predict(final_x_test_data)

In [ ]:
len(y_test.index)

In [ ]:
sns.lineplot(x=y_test.index[win_size:],y=y_test.iloc[win_size:].values, label='Actual')
sns.lineplot(x=y_test.index[win_size:],y=y_pred.ravel(), label='Predicted')
plt.xlabel('Date')
plt.ylabel('Target')
plt.title('Actual vs Predicted')
plt.xticks(rotation=45)
plt.legend()